# 06 Final Evaluation and Ablation Table

Inputs: outputs from notebooks 02, 04, and optionally 05.

This notebook collects all metric JSON/CSV files into one ablation table:
retrieval only, 2.5D Qwen SFT, and 3D dual-stream alignment. It also
stores a clean final summary for the Q1 report.

In [ ]:
from pathlib import Path
import json, warnings
import pandas as pd
warnings.filterwarnings("ignore")

OUTPUT_ROOT = Path("/kaggle/working/vimedpet_q1_outputs") if Path("/kaggle/working").exists() else Path("vimedpet_q1_outputs")
FINAL_DIR = OUTPUT_ROOT / "06_final_eval_ablation"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

search_roots = []
if Path("/kaggle/input").exists(): search_roots.append(Path("/kaggle/input"))
search_roots += [Path.cwd(), OUTPUT_ROOT]

def find_json(name):
    hits = []
    for root in search_roots:
        if root.exists():
            hits += list(root.rglob(name))
    return sorted(hits, key=lambda p: len(str(p)))

rows = []
for p in find_json("retrieval_metrics.json"):
    d = json.loads(p.read_text(encoding="utf-8"))
    rows.append({"model": "2.5D retrieval", "source": str(p), **d})
for p in find_json("qwen_eval_metrics.json"):
    d = json.loads(p.read_text(encoding="utf-8"))
    rows.append({"model": "2.5D Qwen SFT", "source": str(p), **d})
for p in find_json("dual_stream_metrics.json"):
    d = json.loads(p.read_text(encoding="utf-8"))
    rows.append({"model": "3D dual-stream alignment", "source": str(p), **d})

summary = pd.DataFrame(rows)
summary.to_csv(FINAL_DIR / "q1_final_ablation_summary.csv", index=False, encoding="utf-8-sig")
display(summary)

gates = pd.DataFrame([
    {"stage": "retrieval", "metric": "retrieved_rouge_l", "minimum": 0.18},
    {"stage": "retrieval", "metric": "retrieved_keyword_f1", "minimum": 0.25},
    {"stage": "Qwen SFT", "metric": "rouge_l", "minimum": 0.30},
    {"stage": "Qwen SFT", "metric": "bleu4", "minimum": 0.08},
    {"stage": "Qwen SFT", "metric": "keyword_f1", "minimum": 0.30},
    {"stage": "Qwen SFT", "metric": "suv_acc_0_5", "minimum": 0.60},
    {"stage": "3D", "metric": "best_val_loss", "minimum": "decreasing / lower than first epoch"},
])
gates.to_csv(FINAL_DIR / "q1_minimum_gates.csv", index=False, encoding="utf-8-sig")
display(gates)
print("Saved:", FINAL_DIR)